In [8]:
# ==============================
# GPT-2 Fine-Tuning: Recipe Generator
# ==============================

!pip install transformers datasets accelerate -q

import torch, math
from transformers import (
    GPT2LMHeadModel, GPT2Tokenizer,
    Trainer, TrainingArguments,
    DataCollatorForLanguageModeling, set_seed
)
from datasets import Dataset

set_seed(42)

# ==============================
# Load Fresh GPT-2 Model
# ==============================
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2')

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

# ==============================
# Text Generation Function
# ==============================
def generate_text(model, tokenizer, prompt, max_length=120):
    model.eval()
    inputs = tokenizer.encode(prompt, return_tensors='pt').to(model.device)

    with torch.no_grad():
        output = model.generate(
            inputs,
            max_length=max_length,
            temperature=0.8,
            top_k=50,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(output[0], skip_special_tokens=True)

# ==============================
# Baseline Generation
# ==============================
recipe_prompts = [
    "To make butter chicken",
    "For pasta carbonara",
    "To prepare a chocolate cake"
]

print("=== BASELINE RECIPES ===")
baseline = {}

for p in recipe_prompts:
    baseline[p] = generate_text(model, tokenizer, p)
    print(f"\nPrompt: {p}")
    print(f"Output: {baseline[p]}")

# ==============================
# Recipe Dataset
# ==============================
recipes = [
    "to make butter chicken start by marinating chicken pieces in yogurt with turmeric chili powder and garam masala for one hour.",
    "heat butter in a pan and fry onions until golden brown then add ginger garlic paste and cook for two minutes.",
    "add tomato puree and cook on low heat for ten minutes until the oil separates from the masala.",
    "add the marinated chicken and cook on medium heat for fifteen minutes until fully cooked.",
    "finish with fresh cream and kasuri methi and serve hot with naan or steamed rice.",
    "for pasta carbonara boil spaghetti in salted water until al dente and reserve half cup of pasta water.",
    "fry diced pancetta in olive oil until crispy and set aside.",
    "whisk together egg yolks parmesan cheese and black pepper in a bowl.",
    "toss the hot pasta with pancetta and remove from heat then quickly stir in the egg mixture.",
    "the residual heat will cook the eggs into a creamy sauce and serve immediately with extra parmesan.",
    "to prepare vegetable stir fry heat sesame oil in a wok on high heat.",
    "add sliced bell peppers broccoli florets and snap peas and toss for three minutes.",
    "pour in soy sauce oyster sauce and a pinch of sugar and stir well.",
    "add minced garlic and ginger and cook for one more minute until fragrant.",
    "serve the stir fry over steamed jasmine rice and garnish with sesame seeds.",
    "for chocolate chip cookies cream together butter and sugar until light and fluffy.",
    "beat in eggs one at a time then add vanilla extract and mix well.",
    "fold in flour baking soda and salt then gently stir in chocolate chips.",
    "scoop dough onto a baking sheet and bake at 180 degrees for twelve minutes until golden.",
    "let cookies cool on the tray for five minutes before transferring to a wire rack."
]

dataset = Dataset.from_dict({"text": recipes})

# ==============================
# Tokenization
# ==============================
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
split = tokenized.train_test_split(test_size=0.15, seed=42)

# ==============================
# Training Setup
# ==============================
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

training_args = TrainingArguments(
    output_dir="./gpt2-recipes",
    num_train_epochs=15,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=50,
    eval_strategy="epoch", # Changed from evaluation_strategy="epoch"
    logging_steps=10,
    save_strategy="no",
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split["train"],
    eval_dataset=split["test"],
    data_collator=data_collator
)

# ==============================
# Train Model
# ==============================
trainer.train()

# ==============================
# Evaluate Model
# ==============================
eval_results = trainer.evaluate()
perplexity = math.exp(eval_results["eval_loss"])
print(f"\nPerplexity: {perplexity:.2f}")

# ==============================
# Fine-Tuned Generation
# ==============================
print("\n=== FINE-TUNED RECIPES ===")

for p in recipe_prompts:
    ft_output = generate_text(model, tokenizer, p)

    print(f"\nPrompt: {p}")
    print(f"Baseline:   {baseline[p][:120]}")
    print(f"Fine-Tuned: {ft_output[:120]}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


=== BASELINE RECIPES ===

Prompt: To make butter chicken
Output: To make butter chicken, just throw a little at a time. Add the chicken to the bottom of a large Dutch oven. Boil until it reaches 180 degrees. Stir in the milk and spices. Place chicken and chicken stock in a large saucepan. Bring to a boil, then allow to cool. Remove from heat. Reduce heat to medium-low. Cover with a lid and allow to cool. Cook in pan for 5 minutes, or until mixture is soft. Remove from pan, and remove from heat. Serve hot.

Recipe from Easy to Make, by James H. Stroud

Prompt: For pasta carbonara
Output: For pasta carbonara, or as many as I could get them from, would be a good meal for me. I'll probably never have any of them, but I'd like to see a few in the future.

I'm writing to ask you to help me prepare some of the pasta that you have at home.

I'll wait until the pasta is ready before I can get it all together, but I can use your help. I'll include a few images for each of the various pasta that 

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss
1,No log,3.403592
2,3.926337,3.264174
3,3.926337,3.105294
4,3.443312,3.021800
5,3.443312,2.981154
6,2.728940,2.930787
7,2.728940,2.897742
8,2.090287,2.897580
9,2.090287,2.992910
10,1.577087,3.140069



Perplexity: 36.75

=== FINE-TUNED RECIPES ===

Prompt: To make butter chicken
Baseline:   To make butter chicken, just throw a little at a time. Add the chicken to the bottom of a large Dutch oven. Boil until i
Fine-Tuned: To make butter chicken add turmeric chili powder and garam masala and cook for one more minute until the chicken is soft

Prompt: For pasta carbonara
Baseline:   For pasta carbonara, or as many as I could get them from, would be a good meal for me. I'll probably never have any of t
Fine-Tuned: For pasta carbonara boil spaghetti in salted water until al dente and reserve half cup of pasta water. Drain spaghetti f

Prompt: To prepare a chocolate cake
Baseline:   To prepare a chocolate cake, add water to the cake and stir into a well-scraped bowl. You can also add more water if you
Fine-Tuned: To prepare a chocolate cake pan and place the wok on high heat. Add the egg mixture and fry for two minutes until golden
